# A股后复权每日筛选回测（Jupyter）

本 Notebook 调用 `strategy.run_backtest_daily` 执行回测，并直接绘图分析。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'strategy.py').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import strategy

plt.style.use('seaborn-v0_8')

In [ ]:
# 回测参数
START_DATE = '2021-03-01'
END_DATE = '2026-02-26'
TOP_SYMBOLS_PER_DAY = 1   # 每日入选1只=满仓单标的
INITIAL_CASH = 200_000.0

In [ ]:
result, daily_plan, data = strategy.run_backtest_daily(
    start_date=START_DATE,
    end_date=END_DATE,
    top_symbols_per_day=TOP_SYMBOLS_PER_DAY,
    initial_cash=INITIAL_CASH,
)

metrics = result.metrics_df.copy()
metrics

In [ ]:
# 关键指标
metrics.loc[['total_return_pct', 'annualized_return', 'max_drawdown_pct', 'sharpe_ratio', 'trade_count']]

In [ ]:
# 交易记录
result.trades_df.head(20)

In [ ]:
# 净值曲线
equity = result.equity_curve.sort_index()
fig, ax = plt.subplots(figsize=(12, 4))
equity.plot(ax=ax, color='#1f77b4', lw=1.6)
ax.set_title('Equity Curve')
ax.set_xlabel('Date')
ax.set_ylabel('Equity')
ax.grid(alpha=0.25)
plt.show()

In [ ]:
# 回撤曲线
rolling_max = equity.cummax()
drawdown = equity / rolling_max - 1.0
fig, ax = plt.subplots(figsize=(12, 3.5))
drawdown.plot(ax=ax, color='#d62728', lw=1.2)
ax.set_title('Drawdown')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.axhline(0, color='black', lw=0.8)
ax.grid(alpha=0.25)
plt.show()

In [ ]:
# 日收益分布
daily_ret = result.daily_returns
if not isinstance(daily_ret, pd.Series):
    daily_ret = pd.Series(daily_ret)
daily_ret = daily_ret.dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(daily_ret, bins=50, color='#2ca02c', alpha=0.75)
ax.set_title('Daily Return Distribution')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Count')
ax.grid(alpha=0.2)
plt.show()

In [ ]:
# 每日入选标的频次
selected_symbol = pd.Series({d: (syms[0] if syms else None) for d, syms in daily_plan.items()})
selected_symbol.value_counts().head(20)